# CSI across subcarriers

Plots of the raw data of every cluster across subcarriers, antennas, time and the noise-only slot. Reads a subset of frames from `../raw_traces/` and writes figures and tables to `results/`.

## Step 1: Settings

`N_FRAMES` frames per cluster are read, starting at `START_FRAME`.

In [ ]:
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 70)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

def show(df, name, decimals=3):
    df = df.astype({col: "float64" for col in df.select_dtypes("float32").columns}).round(decimals)
    display(df)
    df.to_csv(os.path.join(OUT_DIR, f"{name}.csv"))

RAW_DIR = "../raw_traces"
OUT_DIR = "results"

LOS_CLUSTERS = ["LOS_cluster1", "LOS_cluster2", "LOS_cluster3", "LOS_cluster4"]
NLOS_CLUSTERS = ["NLOS_cluster1", "NLOS_cluster2", "NLOS_cluster3", "NLOS_cluster4", "NLOS_cluster5"]
ALL_CLUSTERS = LOS_CLUSTERS + NLOS_CLUSTERS

START_FRAME = 200
N_FRAMES = 1000
HEATMAP_CLUSTER = "LOS_cluster1"
WATERFALL_CLUSTER = "NLOS_cluster1"
WATERFALL_ANTENNA = 0
TABLE_CLUSTER = "LOS_cluster1"
TABLE_FRAME = 0
TABLE_ANTENNA = 0

SAMPLES_PER_SLOT = 224
LTS_OFFSET = 47
FFT_SIZE = 64
NOISE_SLOT = 4
LTS_FREQ = np.array([0,0,0,0,0,0,1,1,-1,-1,1,1,-1,1,-1,1,1,1,1,1,1,-1,-1,1,1,-1,1,-1,1,1,1,1,
                     0,1,-1,-1,1,1,-1,1,-1,1,-1,-1,-1,-1,-1,1,1,-1,-1,1,-1,1,-1,1,1,1,1,0,0,0,0,0], dtype=np.float32)
NULL_BINS = [0, 1, 2, 3, 4, 5, 32, 59, 60, 61, 62, 63]
DATA_BINS = np.array([b for b in range(FFT_SIZE) if b not in NULL_BINS])
ANT_IDX = np.delete(np.arange(4, 72), [32, 33, 34, 35])

K_ALL = np.arange(FFT_SIZE) - 32
K_DATA = K_ALL[DATA_BINS]

def slots_for(cluster):
    return (0, 3) if cluster.startswith("LOS") else (3, 0)

CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": INK_2, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "lines.linewidth": 2,
})
os.makedirs(OUT_DIR, exist_ok=True)

## Step 2: Load the raw data and take the FFT

Computes the 64-bin spectrum, the calibrated CSI on the 52 data subcarriers and the noise power of the noise-only slot.

In [ ]:
def fft_slot(block, slot, use_lts=True):
    s = slot * SAMPLES_PER_SLOT + LTS_OFFSET
    iq = (block[:, :, s:s + FFT_SIZE, 0] + 1j * block[:, :, s:s + FFT_SIZE, 1]).astype(np.complex64) * 2 ** -15
    h = np.fft.fftshift(np.fft.fft(iq, FFT_SIZE, axis=2), axes=2)
    if use_lts:
        h = h * LTS_FREQ
    return h[:, ::-1, :][:, ANT_IDX, :]

spec64, noise64, csi, summary = {}, {}, {}, []
for c in ALL_CLUSTERS:
    with h5py.File(os.path.join(RAW_DIR, f"channe14_{c}.hdf5"), "r") as f:
        ds = f["Pilot_Samples"]
        stop = ds.shape[0] - 1 if N_FRAMES is None else min(START_FRAME + N_FRAMES, ds.shape[0] - 1)
        block = ds[START_FRAME:stop]
    meas, cal = slots_for(c)
    raw = fft_slot(block, meas, use_lts=False)
    spec64[c] = (np.abs(raw) ** 2).mean(axis=(0, 1))
    h_meas = fft_slot(block, meas)[:, :, DATA_BINS]
    h_cal = fft_slot(block, cal)[:, :, DATA_BINS]
    csi[c] = h_meas / (h_cal / np.abs(h_cal))
    noise64[c] = (np.abs(fft_slot(block, NOISE_SLOT, use_lts=False)) ** 2).mean(axis=(0, 1))
    if c == TABLE_CLUSTER:
        raw_block = block[TABLE_FRAME]
    summary.append({"cluster": c, "class": "LOS" if c.startswith("LOS") else "NLOS",
                    "first frame": START_FRAME, "last frame": stop - 1, "frames loaded": len(block),
                    "CSI shape": str(csi[c].shape),
                    "mean |H| (dB)": 20 * np.log10(np.abs(csi[c]).mean())})
    del block, raw, h_meas, h_cal

df = pd.DataFrame(summary).set_index("cluster")
show(df, "loaded_clusters", 2)

## Step 3: Raw data as tables

Raw I/Q samples, all 64 FFT bins and the CSI on the 52 data subcarriers for one frame and one antenna.

In [ ]:
meas, cal = slots_for(TABLE_CLUSTER)
ant_channel = ANT_IDX[TABLE_ANTENNA]
radio = raw_block.shape[0] - 1 - ant_channel
s0 = meas * SAMPLES_PER_SLOT + LTS_OFFSET
I = raw_block[radio, s0:s0 + FFT_SIZE, 0].astype(int)
Q = raw_block[radio, s0:s0 + FFT_SIZE, 1].astype(int)

df = pd.DataFrame({"sample in file": np.arange(s0, s0 + FFT_SIZE), "I (int16)": I, "Q (int16)": Q,
                   "|I + jQ| (scaled)": np.abs((I + 1j * Q) * 2 ** -15)})
df.index.name = "pilot sample"
print(f"1) raw pilot samples: {TABLE_CLUSTER}, frame {START_FRAME + TABLE_FRAME}, antenna {TABLE_ANTENNA} "
      f"(radio channel {radio}), slot {meas}")
show(df, "raw_pilot_samples", 5)

x = (I + 1j * Q).astype(np.complex64) * 2 ** -15
X = np.fft.fftshift(np.fft.fft(x, FFT_SIZE))
df = pd.DataFrame({"802.11 k": K_ALL, "kept": [b in DATA_BINS for b in range(FFT_SIZE)],
                   "|FFT|": np.abs(X), "|FFT| (dB)": 20 * np.log10(np.abs(X) + 1e-12)})
df.index.name = "FFT bin"
print("2) all 64 FFT bins of these samples")
show(df, "raw_fft_64_bins", 4)

h = csi[TABLE_CLUSTER][TABLE_FRAME, TABLE_ANTENNA]
df = pd.DataFrame({"802.11 k": K_DATA, "Re(H)": h.real, "Im(H)": h.imag, "|H|": np.abs(h),
                   "|H| (dB)": 20 * np.log10(np.abs(h)), "phase (rad)": np.angle(h),
                   "|H| / antenna mean": np.abs(h) / np.abs(h).mean()})
df.index.name = "model column"
print("3) calibrated CSI on the 52 data subcarriers")
show(df, "csi_52_subcarriers", 4)

## Step 4: Full 64-bin spectrum

The 12 shaded bins (guard bands and DC) carry no channel information and are removed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, group, title in [(axes[0], LOS_CLUSTERS, "LOS clusters"), (axes[1], NLOS_CLUSTERS, "NLOS clusters")]:
    ax.set_axisbelow(True)
    for b in NULL_BINS:
        ax.axvspan(K_ALL[b] - 0.5, K_ALL[b] + 0.5, color=GRID, alpha=0.8, linewidth=0)
    for i, c in enumerate(group):
        p = spec64[c]
        ax.plot(K_ALL, 10 * np.log10(p / p.max()), color=CAT[i], label=c.replace("_cluster", " "))
    ax.set_xlabel("802.11 subcarrier index k (shaded = removed)")
    ax.set_title(f"{title}: all 64 FFT bins")
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("Power relative to strongest bin (dB)")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "spectrum_64_bins.png"), dpi=150)
plt.show()

df = pd.DataFrame({c.replace("_cluster", " "): 10 * np.log10(spec64[c] / spec64[c].max()) for c in ALL_CLUSTERS})
df.insert(0, "802.11 k", K_ALL)
df.insert(1, "kept", [b in DATA_BINS for b in range(FFT_SIZE)])
df.index.name = "FFT bin"
print("power per FFT bin, relative to the strongest bin (dB)")
show(df, "spectrum_64_bins", 2)

## Step 5: Amplitude of the 52 data subcarriers

Absolute levels include the different receive gains of the LOS and NLOS recordings, so compare the shape, not the level.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, group, title in [(axes[0], LOS_CLUSTERS, "LOS clusters"), (axes[1], NLOS_CLUSTERS, "NLOS clusters")]:
    for i, c in enumerate(group):
        a = np.abs(csi[c]).mean(axis=(0, 1))
        ax.plot(K_DATA, 20 * np.log10(a), "o-", color=CAT[i], markersize=3, linewidth=1.5, label=c.replace("_cluster", " "))
    ax.set_xlabel("802.11 subcarrier index k")
    ax.set_ylabel("Mean |H| (dB)")
    ax.set_title(f"{title}: mean amplitude per subcarrier")
    ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "amplitude_per_subcarrier.png"), dpi=150)
plt.show()

df = pd.DataFrame({c.replace("_cluster", " "): 20 * np.log10(np.abs(csi[c]).mean(axis=(0, 1))) for c in ALL_CLUSTERS})
df.insert(0, "802.11 k", K_DATA)
df.index.name = "model column"
print("mean |H| per subcarrier (dB), averaged over frames and antennas")
show(df, "amplitude_per_subcarrier", 2)

## Step 6: Per-antenna normalised amplitude

This is the model input. A flat profile indicates little frequency selectivity, ripples indicate multipath.

In [ ]:
norm = {c: np.abs(csi[c]) / np.abs(csi[c]).mean(axis=2, keepdims=True) for c in ALL_CLUSTERS}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, group, title in [(axes[0], LOS_CLUSTERS, "LOS clusters"), (axes[1], NLOS_CLUSTERS, "NLOS clusters")]:
    ax.axhline(1.0, color=INK_2, linestyle="--", linewidth=1)
    for i, c in enumerate(group):
        ax.plot(K_DATA, norm[c].mean(axis=(0, 1)), "o-", color=CAT[i], markersize=3, linewidth=1.5,
                label=c.replace("_cluster", " "))
    ax.set_xlabel("802.11 subcarrier index k")
    ax.set_title(f"{title}: per-antenna normalised amplitude")
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("|H| / antenna mean")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "normalised_per_subcarrier.png"), dpi=150)
plt.show()

df = pd.DataFrame({c.replace("_cluster", " "): norm[c].mean(axis=(0, 1)) for c in ALL_CLUSTERS})
df.insert(0, "802.11 k", K_DATA)
df.index.name = "model column"
print("per-antenna normalised |H| per subcarrier (mean over frames and antennas)")
show(df, "normalised_per_subcarrier", 3)

rows = []
for c in ALL_CLUSTERS:
    a = np.abs(csi[c])
    cv = a.std(axis=2) / a.mean(axis=2)
    rows.append({"cluster": c, "class": "LOS" if c.startswith("LOS") else "NLOS",
                 "CV median": np.median(cv), "CV 10th pct": np.percentile(cv, 10), "CV 90th pct": np.percentile(cv, 90)})
df = pd.DataFrame(rows).set_index("cluster")
print("frequency selectivity: coefficient of variation of |H| across the 52 subcarriers")
show(df, "frequency_selectivity", 3)

## Step 7: Antenna × subcarrier heatmap

In [ ]:
A = 20 * np.log10(np.abs(csi[HEATMAP_CLUSTER]).mean(axis=0))
fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.imshow(A, aspect="auto", cmap="Blues", origin="lower",
               extent=[-0.5, 51.5, -0.5, 63.5], interpolation="nearest")
ax.grid(False)
tick_cols = np.arange(0, 52, 6)
ax.set_xticks(tick_cols, [str(k) for k in K_DATA[tick_cols]])
ax.set_xlabel("802.11 subcarrier index k")
ax.set_ylabel("Antenna (0–63)")
ax.set_title(f"{HEATMAP_CLUSTER.replace('_cluster', ' ')}: mean |H| per antenna and subcarrier (dB)")
fig.colorbar(im, ax=ax, label="dB")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f"heatmap_{HEATMAP_CLUSTER}.png"), dpi=150)
plt.show()

## Step 8: One antenna over time

In [ ]:
Wf = norm[WATERFALL_CLUSTER][:, WATERFALL_ANTENNA, :]
fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.imshow(Wf, aspect="auto", cmap="Blues", origin="lower", interpolation="nearest",
               extent=[-0.5, 51.5, START_FRAME, START_FRAME + len(Wf)])
ax.grid(False)
ax.set_xticks(tick_cols, [str(k) for k in K_DATA[tick_cols]])
ax.set_xlabel("802.11 subcarrier index k")
ax.set_ylabel("Frame")
ax.set_title(f"{WATERFALL_CLUSTER.replace('_cluster', ' ')}, antenna {WATERFALL_ANTENNA}: normalised |H| over time")
fig.colorbar(im, ax=ax, label="|H| / antenna mean")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f"waterfall_{WATERFALL_CLUSTER}_ant{WATERFALL_ANTENNA}.png"), dpi=150)
plt.show()

## Step 9: Noise-only slot per subcarrier

The peak around k = −13 is a narrow-band interference component, stronger in the NLOS recordings.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, group, title in [(axes[0], LOS_CLUSTERS, "LOS clusters"), (axes[1], NLOS_CLUSTERS, "NLOS clusters")]:
    ax.axhline(0, color=INK_2, linestyle="--", linewidth=1)
    for i, c in enumerate(group):
        p = noise64[c][DATA_BINS]
        ax.plot(K_DATA, 10 * np.log10(p / np.median(p)), "o-", color=CAT[i], markersize=3, linewidth=1.5,
                label=c.replace("_cluster", " "))
    ax.set_xlabel("802.11 subcarrier index k")
    ax.set_title(f"{title}: noise slot per subcarrier")
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("Noise power vs median subcarrier (dB)")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "noise_per_subcarrier.png"), dpi=150)
plt.show()

df = pd.DataFrame({c.replace("_cluster", " "): 10 * np.log10(noise64[c][DATA_BINS] / np.median(noise64[c][DATA_BINS]))
                   for c in ALL_CLUSTERS})
df.insert(0, "802.11 k", K_DATA)
df.index.name = "model column"
print("noise-slot power per subcarrier vs the median subcarrier (dB)")
show(df, "noise_per_subcarrier", 2)

spur = df[df[[c.replace("_cluster", " ") for c in ALL_CLUSTERS]].max(axis=1) > 3]
print("subcarriers more than 3 dB above the median in at least one cluster (possible spurs)")
show(spur, "noise_spur_subcarriers", 2)

## Step 10: Distribution per subcarrier, LOS against NLOS

Median and 10th to 90th percentile of the normalised amplitude.

In [ ]:
pooled = {"LOS": np.concatenate([norm[c].reshape(-1, 52) for c in LOS_CLUSTERS]),
          "NLOS": np.concatenate([norm[c].reshape(-1, 52) for c in NLOS_CLUSTERS])}
fig, ax = plt.subplots(figsize=(12, 4.4))
for name, color in [("LOS", CAT[0]), ("NLOS", CAT[1])]:
    q10, q50, q90 = np.percentile(pooled[name], [10, 50, 90], axis=0)
    ax.fill_between(K_DATA, q10, q90, color=color, alpha=0.2, linewidth=0)
    ax.plot(K_DATA, q50, "o-", color=color, markersize=3, linewidth=1.5, label=f"{name}: median (band = 10th–90th percentile)")
ax.axhline(1.0, color=INK_2, linestyle="--", linewidth=1)
ax.set_xlabel("802.11 subcarrier index k")
ax.set_ylabel("|H| / antenna mean")
ax.set_title("Per-antenna normalised amplitude per subcarrier: LOS vs NLOS (all clusters)")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "distribution_per_subcarrier.png"), dpi=150)
plt.show()

cols = {}
for name in ["LOS", "NLOS"]:
    q10, q50, q90 = np.percentile(pooled[name], [10, 50, 90], axis=0)
    cols.update({f"{name} p10": q10, f"{name} median": q50, f"{name} p90": q90})
df = pd.DataFrame(cols)
df.insert(0, "802.11 k", K_DATA)
df.index.name = "model column"
print("per-antenna normalised |H| per subcarrier: percentiles over all frames and antennas of each class")
show(df, "distribution_per_subcarrier", 3)

print("figures and CSV tables saved in", os.path.abspath(OUT_DIR))